In [3]:
# ============================================================
# CELL 1 — THERMISTOR VALIDATION SETUP
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# Thermistor validation configuration
# ------------------------------------------------------------

THERMISTOR_CONFIG = {

    "sensor_type": "thermistor",

    "purpose": (
        "Auxiliary respiratory waveform monitoring "
        "and hardware validation"
    ),

    "ml_input": False,

    "validation_targets": [
        "raw waveform acquisition",
        "signal stability",
        "respiratory waveform visibility",
        "respiratory peak detection",
        "respiratory rate estimation",
        "respiratory pause detection",
        "signal quality"
    ],

    "sampling_rate_hz": None,

    "adc_resolution_bits": None,

    "adc_reference_voltage": None,

    "filtering": {
        "low_cutoff_hz": 0.03,
        "high_cutoff_hz": 2.0,
        "filter_order": 4
    },

    "status": "hardware validation setup"
}


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("THERMISTOR VALIDATION — SETUP")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nPurpose:")
print(
    THERMISTOR_CONFIG["purpose"]
)

print("\nML input:")
print(
    THERMISTOR_CONFIG["ml_input"]
)

print("\nValidation targets:")

for i, target in enumerate(
    THERMISTOR_CONFIG["validation_targets"],
    start=1
):
    print(f"{i}. {target}")

print("\nInitial respiration filter:")
print(
    THERMISTOR_CONFIG["filtering"]
)

print(
    "\nPASS: Thermistor validation environment initialized."
)

THERMISTOR VALIDATION — SETUP

Project root:
c:\Users\Mohamed Rizwan M J\Desktop\neonatal-apnea-monitor

Purpose:
Auxiliary respiratory waveform monitoring and hardware validation

ML input:
False

Validation targets:
1. raw waveform acquisition
2. signal stability
3. respiratory waveform visibility
4. respiratory peak detection
5. respiratory rate estimation
6. respiratory pause detection
7. signal quality

Initial respiration filter:
{'low_cutoff_hz': 0.03, 'high_cutoff_hz': 2.0, 'filter_order': 4}

PASS: Thermistor validation environment initialized.


In [1]:
# ============================================================
# CELL 2 — THERMISTOR DIVIDER CIRCUIT VERIFICATION
# ============================================================

print("=" * 70)
print("CELL 2 — THERMISTOR DIVIDER CIRCUIT VERIFICATION")
print("=" * 70)

# Hardware parameters
SUPPLY_VOLTAGE = 3.3
FIXED_RESISTOR_OHMS = 10_000.0

NTC_R25_OHMS = 10_000.0
NTC_BETA_K = 3950.0
REFERENCE_TEMPERATURE_C = 25.0

# ADS1115 measurement range selected for design verification
ADS1115_FULL_SCALE_VOLTAGE = 4.096


# ------------------------------------------------------------
# Divider calculation
# ------------------------------------------------------------
#       3.3 V
#         |
#       10 kΩ
#         |
#         +------> ADS1115 A0
#         |
#       10 kΩ NTC
#         |
#        GND
#
# Vout = Vsupply × R_NTC / (R_fixed + R_NTC)
# ------------------------------------------------------------

VOUT_25C = (
    SUPPLY_VOLTAGE
    * NTC_R25_OHMS
    / (FIXED_RESISTOR_OHMS + NTC_R25_OHMS)
)


# Divider current
DIVIDER_CURRENT_A = (
    SUPPLY_VOLTAGE
    / (FIXED_RESISTOR_OHMS + NTC_R25_OHMS)
)

DIVIDER_CURRENT_uA = DIVIDER_CURRENT_A * 1_000_000


# Power dissipation
FIXED_RESISTOR_POWER_W = (
    DIVIDER_CURRENT_A ** 2
    * FIXED_RESISTOR_OHMS
)

NTC_POWER_W = (
    DIVIDER_CURRENT_A ** 2
    * NTC_R25_OHMS
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nThermistor configuration:")
print(f"NTC R25              : {NTC_R25_OHMS / 1000:.1f} kΩ")
print(f"NTC Beta              : {NTC_BETA_K:.0f} K")
print(f"Reference temperature : {REFERENCE_TEMPERATURE_C:.1f} °C")

print("\nCircuit:")
print(f"Supply voltage        : {SUPPLY_VOLTAGE:.2f} V")
print(f"Fixed resistor        : {FIXED_RESISTOR_OHMS / 1000:.1f} kΩ")

print("\nAt 25 °C:")
print(f"NTC resistance        : {NTC_R25_OHMS:.1f} Ω")
print(f"Divider midpoint      : {VOUT_25C:.4f} V")
print(f"Divider current       : {DIVIDER_CURRENT_uA:.2f} µA")
print(
    f"Fixed resistor power  : "
    f"{FIXED_RESISTOR_POWER_W * 1000:.4f} mW"
)
print(
    f"NTC power             : "
    f"{NTC_POWER_W * 1000:.4f} mW"
)


# ------------------------------------------------------------
# ADC input verification
# ------------------------------------------------------------

print("\nADS1115 design check:")
print(
    f"Expected A0 voltage  : {VOUT_25C:.4f} V"
)
print(
    f"Selected FSR         : ±{ADS1115_FULL_SCALE_VOLTAGE:.3f} V"
)

assert VOUT_25C > 0
assert VOUT_25C < ADS1115_FULL_SCALE_VOLTAGE

print("\nPASS: Thermistor divider calculation completed.")
print(
    "PASS: 25 °C divider voltage is within "
    "the selected measurement range."
)

print(
    "\nNOTE: Actual thermistor waveform acquisition "
    "will be tested only after hardware assembly."
)

CELL 2 — THERMISTOR DIVIDER CIRCUIT VERIFICATION

Thermistor configuration:
NTC R25              : 10.0 kΩ
NTC Beta              : 3950 K
Reference temperature : 25.0 °C

Circuit:
Supply voltage        : 3.30 V
Fixed resistor        : 10.0 kΩ

At 25 °C:
NTC resistance        : 10000.0 Ω
Divider midpoint      : 1.6500 V
Divider current       : 165.00 µA
Fixed resistor power  : 0.2723 mW
NTC power             : 0.2723 mW

ADS1115 design check:
Expected A0 voltage  : 1.6500 V
Selected FSR         : ±4.096 V

PASS: Thermistor divider calculation completed.
PASS: 25 °C divider voltage is within the selected measurement range.

NOTE: Actual thermistor waveform acquisition will be tested only after hardware assembly.


In [4]:
# ============================================================
# CELL 3 — NTC TEMPERATURE-RESISTANCE MODEL
# ============================================================

print("=" * 70)
print("CELL 3 — NTC TEMPERATURE-RESISTANCE MODEL")
print("=" * 70)

# ------------------------------------------------------------
# NTC Beta equation
#
# R(T) = R25 × exp[B × (1/T - 1/T25)]
#
# T and T25 are in Kelvin
# ------------------------------------------------------------

def ntc_resistance(temperature_c):
    temperature_k = temperature_c + 273.15
    reference_temperature_k = REFERENCE_TEMPERATURE_C + 273.15

    resistance = (
        NTC_R25_OHMS
        * np.exp(
            NTC_BETA_K
            * (
                (1 / temperature_k)
                - (1 / reference_temperature_k)
            )
        )
    )

    return resistance


# ------------------------------------------------------------
# Divider voltage
# ------------------------------------------------------------

def divider_voltage(ntc_resistance_ohms):
    return (
        SUPPLY_VOLTAGE
        * ntc_resistance_ohms
        / (FIXED_RESISTOR_OHMS + ntc_resistance_ohms)
    )


# ------------------------------------------------------------
# Evaluate representative temperatures
# ------------------------------------------------------------

temperatures_c = np.array([
    15.0,
    20.0,
    25.0,
    30.0,
    35.0,
    40.0,
    45.0
])

resistances_ohms = np.array([
    ntc_resistance(t)
    for t in temperatures_c
])

voltages_v = np.array([
    divider_voltage(r)
    for r in resistances_ohms
])


# ------------------------------------------------------------
# Create result table
# ------------------------------------------------------------

thermistor_table = pd.DataFrame({
    "Temperature_C": temperatures_c,
    "NTC_Resistance_kOhm": resistances_ohms / 1000,
    "Divider_Voltage_V": voltages_v
})

print("\nTheoretical NTC response:")
print(thermistor_table.to_string(index=False))


# ------------------------------------------------------------
# Verify expected NTC behavior
# ------------------------------------------------------------

# NTC resistance must decrease as temperature increases
resistance_decreases = np.all(
    np.diff(resistances_ohms) < 0
)

# Divider voltage also decreases because the NTC is the
# lower resistor in the divider.
voltage_decreases = np.all(
    np.diff(voltages_v) < 0
)

assert resistance_decreases
assert voltage_decreases


# Verify 25°C result
index_25 = np.where(temperatures_c == 25.0)[0][0]

assert abs(
    resistances_ohms[index_25] - NTC_R25_OHMS
) < 1e-6

assert abs(
    voltages_v[index_25] - VOUT_25C
) < 1e-9


print("\nPASS: NTC resistance decreases with temperature.")
print("PASS: Divider voltage decreases with temperature.")
print("PASS: 25 °C resistance matches 10 kΩ.")
print("PASS: 25 °C divider voltage matches 1.6500 V.")

print(
    "\nNOTE: These are theoretical temperature-response values. "
    "They are not yet respiratory measurements."
)

CELL 3 — NTC TEMPERATURE-RESISTANCE MODEL

Theoretical NTC response:
 Temperature_C  NTC_Resistance_kOhm  Divider_Voltage_V
          15.0            15.837148           2.022769
          20.0            12.535326           1.835632
          25.0            10.000000           1.650000
          30.0             8.037141           1.470442
          35.0             6.505531           1.300670
          40.0             5.301467           1.143344
          45.0             4.348137           1.000050

PASS: NTC resistance decreases with temperature.
PASS: Divider voltage decreases with temperature.
PASS: 25 °C resistance matches 10 kΩ.
PASS: 25 °C divider voltage matches 1.6500 V.

NOTE: These are theoretical temperature-response values. They are not yet respiratory measurements.


In [5]:
# ============================================================
# CELL 4 — HARDWARE ASSEMBLY VERIFICATION CHECKLIST
# ============================================================

print("=" * 70)
print("CELL 4 — THERMISTOR HARDWARE ASSEMBLY CHECKLIST")
print("=" * 70)

# ------------------------------------------------------------
# Finalized hardware configuration
# ------------------------------------------------------------

HARDWARE_CONFIG = {
    "sensor": "MF52A103J3950 NTC thermistor",
    "ntc_resistance_25c_ohms": 10_000,
    "ntc_beta_k": 3950,
    "ntc_tolerance": "±5%",
    
    "fixed_resistor_ohms": 10_000,
    "supply_voltage_v": 3.3,
    
    "adc": "ADS1115",
    "adc_resolution_bits": 16,
    "adc_channels": 4,
    "adc_input_channel": "A0",
    "adc_interface": "I2C",
    
    "controller": "Raspberry Pi 4",
    
    "circuit": (
        "3.3V -> 10k fixed resistor -> "
        "divider midpoint -> 10k NTC -> GND"
    )
}


# ------------------------------------------------------------
# Assembly checklist
# ------------------------------------------------------------

ASSEMBLY_CHECKLIST = [
    ("NTC identified", True),
    ("NTC is MF52A103J3950", True),
    ("NTC nominal resistance is 10 kΩ @ 25°C", True),
    ("NTC Beta value is 3950 K", True),
    ("Fixed resistor is 10 kΩ", True),
    ("Supply planned as 3.3 V", True),
    ("Divider midpoint connected to ADS1115 A0", True),
    ("ADS1115 planned for I2C communication", True),
    ("ADS1115 connected to Raspberry Pi 4", True),
    ("Common ground planned", True),
]


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("\nFINAL HARDWARE CONFIGURATION")
print("-" * 70)

for key, value in HARDWARE_CONFIG.items():
    print(f"{key}: {value}")


print("\nASSEMBLY CHECKLIST")
print("-" * 70)

for item, status in ASSEMBLY_CHECKLIST:
    print(
        f"[{'READY' if status else 'CHECK'}] {item}"
    )


# ------------------------------------------------------------
# Verify configuration
# ------------------------------------------------------------

assert HARDWARE_CONFIG["sensor"] == (
    "MF52A103J3950 NTC thermistor"
)

assert HARDWARE_CONFIG["ntc_resistance_25c_ohms"] == 10_000

assert HARDWARE_CONFIG["ntc_beta_k"] == 3950

assert HARDWARE_CONFIG["fixed_resistor_ohms"] == 10_000

assert HARDWARE_CONFIG["supply_voltage_v"] == 3.3

assert HARDWARE_CONFIG["adc"] == "ADS1115"

assert HARDWARE_CONFIG["adc_input_channel"] == "A0"

assert HARDWARE_CONFIG["controller"] == "Raspberry Pi 4"


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PASS: Hardware configuration verified.")
print("PASS: Thermistor divider topology verified.")
print("PASS: ADS1115 A0 is the planned measurement point.")
print("PASS: Raspberry Pi 4 is the planned edge controller.")
print("=" * 70)

print(
    "\nSTATUS: Hardware assembly has NOT started."
)

print(
    "NEXT STEP AFTER PHYSICAL ASSEMBLY: "
    "ADS1115 I2C communication and raw ADC acquisition."
)

CELL 4 — THERMISTOR HARDWARE ASSEMBLY CHECKLIST

FINAL HARDWARE CONFIGURATION
----------------------------------------------------------------------
sensor: MF52A103J3950 NTC thermistor
ntc_resistance_25c_ohms: 10000
ntc_beta_k: 3950
ntc_tolerance: ±5%
fixed_resistor_ohms: 10000
supply_voltage_v: 3.3
adc: ADS1115
adc_resolution_bits: 16
adc_channels: 4
adc_input_channel: A0
adc_interface: I2C
controller: Raspberry Pi 4
circuit: 3.3V -> 10k fixed resistor -> divider midpoint -> 10k NTC -> GND

ASSEMBLY CHECKLIST
----------------------------------------------------------------------
[READY] NTC identified
[READY] NTC is MF52A103J3950
[READY] NTC nominal resistance is 10 kΩ @ 25°C
[READY] NTC Beta value is 3950 K
[READY] Fixed resistor is 10 kΩ
[READY] Supply planned as 3.3 V
[READY] Divider midpoint connected to ADS1115 A0
[READY] ADS1115 planned for I2C communication
[READY] ADS1115 connected to Raspberry Pi 4
[READY] Common ground planned

PASS: Hardware configuration verified.
PASS: T